# 05 – Clustering: Segmentierung der Antragsteller

**Unüberwachtes** Verfahren (Teilfrage 4). **TARGET wird NICHT verwendet** – sonst wäre
die Segmentierung zirkulär. Die Ausfallrate je Cluster betrachten wir erst **nachträglich**
als externe Validierung, nicht als Trainingssignal.

**Warum Clustering überhaupt:**  
Klassifikationsmodelle beantworten *„Fällt dieser Antragsteller aus?"*, Clustering beantwortet *„Welche Typen von Antragstellern gibt es?"*. Diese Frage ist für das Risikomanagement eigenständig wertvoll:
- **Produktgestaltung:** Für bestimmte Kundensegmente können passende Kreditprodukte entwickelt werden (z. B. kleinere Raten für jüngere Kreditnehmer).
- **Gezielte Beratung:** Hochrisikogruppen können proaktiv angesprochen werden.
- **Erklärbares Risikomanagement:** Segmente sind intuitiv kommunizierbar – im Gegensatz zu Black-Box-Modellscores.

**Vorgehen in diesem Notebook:**  
1. Feature-Auswahl und Skalierung (ohne TARGET)  
2. Optimale Clusterzahl bestimmen (Elbow + Silhouette)  
3. K-Means trainieren und Cluster in 2D visualisieren (PCA)  
4. Cluster-Profile erstellen und nachträgliche Ausfallraten analysieren  
5. Fachliche Benennung und Interpretation der Segmente

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, utils, clustering as cl
feat = utils.load_processed("feature_matrix")

## 1. Feature-Auswahl (ohne TARGET) & Skalierung

**Warum diese 12 Merkmale?**  
Die ausgewählten Merkmale decken die relevantesten Dimensionen eines Kreditnehmerprofils ab:

| Merkmal | Dimension | Begründung |
|---|---|---|
| `AMT_INCOME_TOTAL`, `AMT_CREDIT` | Wirtschaftliche Stärke | Absolute Größenordnung des Kreditverhältnisses |
| `credit_income_ratio`, `annuity_income_ratio` | Verschuldungsgrad | Gibt an, wie stark das Einkommen durch Kreditverpflichtungen belastet ist |
| `age_years`, `employment_years` | Lebenssituation | Stabilität und Lebensphase des Antragstellers |
| `external_score_mean` | Bonität | Aggregierter externer Kreditwürdigkeitsscore |
| `BUREAU_COUNT`, `BUREAU_ACTIVE_COUNT` | Kredit-Historie | Umfang bisheriger Kreditaktivität |
| `PREV_COUNT` | Antragserfahrung | Anzahl früherer Anträge bei Home Credit |
| `INST_LATE_PAYMENT_RATIO` | Zahlungsverhalten | Anteil verspäteter Raten – direktes Risikomerkmal |
| `missing_values_count` | Informationsqualität | Kreditnehmer mit vielen fehlenden Angaben bilden ggf. eine eigene Gruppe |

**Warum Skalierung zwingend notwendig ist:**  
K-Means misst Ähnlichkeit über euklidische Distanzen. `AMT_INCOME_TOTAL` hat Werte in Hunderttausenden, `INST_LATE_PAYMENT_RATIO` liegt zwischen 0 und 1 – ohne Standardisierung würde das Einkommen den Abstandsterm vollständig dominieren und alle anderen Merkmale effektiv ignoriert. Der StandardScaler normiert jedes Merkmal auf Mittelwert 0 und Standardabweichung 1, sodass alle Dimensionen gleichgewichtet eingehen.

**Ergebnis:** 307 511 Antragsteller × 12 Merkmale (vollständig skaliert, keine NaN).

In [2]:
X = cl.select_cluster_features(feat)
pre = cl.build_clustering_preprocessor()
X_scaled = pre.fit_transform(X)
print("Clustering-Matrix:", X_scaled.shape)

22:36:12 | INFO    | Clustering nutzt 12 Merkmale: ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'credit_income_ratio', 'annuity_income_ratio', 'age_years', 'employment_years', 'external_score_mean', 'BUREAU_COUNT', 'BUREAU_ACTIVE_COUNT', 'PREV_COUNT', 'INST_LATE_PAYMENT_RATIO', 'missing_values_count']


Clustering-Matrix: (307511, 12)


## 2. Wahl der Clusterzahl: Elbow & Silhouette

**Wie funktionieren die Kriterien?**
- **Elbow (Inertia):** Misst die Summe der quadrierten Abstände jedes Punktes zu seinem Cluster-Zentroid. Mit wachsendem k sinkt die Inertia zwangsläufig – gesucht wird der Punkt, ab dem der Gewinn abnimmt (der „Knick").
- **Silhouette-Score:** Vergleicht für jeden Punkt, wie ähnlich er seinem eigenen Cluster ist (Kohäsion) vs. dem nächsten fremden Cluster (Separation). Werte nahe 1 = gut getrennte Cluster; nahe 0 = Punkte liegen an der Grenze; negativ = falsch zugeordnet.

**Konkrete Ergebnisse:**

| k | Inertia | Silhouette |
|---|---|---|
| 2 | 3 234 816 | **0.147** ← Maximum |
| 3 | 2 923 329 | 0.142 |
| 4 | 2 680 696 | 0.142 |
| 5 | 2 493 126 | 0.107 |
| 6 | 2 351 221 | 0.110 |
| 7 | 2 252 293 | 0.102 |
| 8 | 2 132 296 | 0.108 |

### Interpretation der grafischen & tabellarischen Ergebnisse:
* **Fehlender „Knick“ in der Elbow-Methode:** Ein Blick auf das linke Diagramm zeigt einen sehr gleichmäßigen, fast linearen Abfall der Inertia. Es gibt keinen prägnanten „Elbow“ (Knick), der uns eine klare mathematische Empfehlung für die Clusterzahl geben würde. Dies macht die zusätzliche Betrachtung der Silhouette-Analyse zwingend erforderlich.
* **Plateau und Absturz im Silhouette-Score:** Das rechte Diagramm zeigt das Maximum bei k=2. Danach bildet sich ein auffälliges kleines Plateau: Für k=3 und k=4 bleibt der Score stabil bei etwa 0.142. Ab k=5 stürzt der Wert jedoch steil ab auf unter 0.11. Das zeigt grafisch sehr deutlich, dass eine Aufteilung in 5 oder mehr Cluster die Datenstruktur künstlich zerschneidet und die Qualität drastisch mindert.
* **Generell niedrige Silhouette-Werte:** Die Werte sind insgesamt recht niedrig (max. 0.147). Das ist typisch für reale sozioökonomische Daten: Kreditnehmerprofile gehen *graduell* ineinander über – es gibt keine scharfen „natürlichen" Grenzen. Niedrige Silhouette-Werte bedeuten nicht, dass das Clustering wertlos ist, sondern dass die Segmente eher weiche Übergangszonen als hart getrennte, diskrete Klassen darstellen.

**Warum dennoch k=4?**  
Das Silhouette-Optimum bei k=2 liefert nur zwei sehr grobe Segmente mit begrenztem Informationsgehalt. k=3 und k=4 haben nahezu identische Silhouette-Werte (0.142), bieten aber deutlich mehr fachliche Differenzierung. k=4 wurde als Kompromiss zwischen statistischer Qualität und inhaltlicher Interpretierbarkeit gewählt – eine gängige Praxis in der angewandten Segmentierungsanalyse.

In [3]:
best_k, inertias, silhouettes = cl.find_optimal_k(X_scaled, k_range=range(2, 9))
plt.show()
print("Vorgeschlagenes k (Silhouette):", best_k)

22:36:38 | INFO    | Silhouette-Subsampling: 307511 → 8000 Zeilen (KMeans auf vollen Daten)


c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\dev\Lib\subproces

Vorgeschlagenes k (Silhouette): 2


C:\Users\annis\AppData\Local\Temp\ipykernel_12020\2732511711.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. K-Means & PCA-Visualisierung

**Warum PCA zur Visualisierung?**  
Die Cluster-Matrix hat 12 Dimensionen – direkt visualisierbar ist das nicht. PCA (Principal Component Analysis) komprimiert die 12 Merkmale auf die 2 Richtungen maximaler Varianz (PC1, PC2). Dadurch können wir prüfen, ob die Cluster im Merkmalsraum räumlich getrennt liegen.

**Was bedeuten die PCA-Achsen?**  
PC1 und PC2 sind Linearkombinationen der Originalmerkmale. Typischerweise fasst PC1 die stärkste Gemeinsamkeit zusammen (hier vermutlich allgemeines Kreditvolumen/Bonität), PC2 eine davon unabhängige zweite Dimension (z. B. Alter vs. Schuldenstruktur). Die genaue Interpretation der Achsen erfordert eine Analyse der Ladungen, ist aber für die visuelle Prüfung nicht zwingend nötig.

### Interpretation der Grafik

* **Erklärte Varianz:** Die beiden abgebildeten Hauptkomponenten erklären zusammen **35,3 % der Gesamtvarianz** (19,4 % in PC1 + 15,9 % in PC2). Das bedeutet im Umkehrschluss: Knapp 65 % der trennenden Informationen liegen in höheren Dimensionen verborgen, die dieser Plot *nicht* zeigt. Überlappende Cluster in dieser 2D-Ansicht sind daher **erwartbar und nicht problematisch** – die eigentliche K-Means-Zuordnung erfolgte trennscharf im vollen 12-dimensionalen Raum.
* **Charakteristik der sichtbaren Cluster:**
  * **Cluster 0 (Blau):** Streut stark in den positiven Bereich der PC1-Achse. Diese Gruppe hebt sich entlang der ersten Hauptkomponente deutlich von den anderen ab (z. B. Kunden mit deutlich höheren Kreditbeträgen).
  * **Cluster 1 (Braun):** Zieht sich sehr weit nach oben entlang der positiven PC2-Achse. Hier gibt es einige deutliche Ausreißer am oberen und linken Rand, was auf eine höhere Varianz innerhalb dieser Gruppe hindeutet.
  * **Cluster 2 (Türkis):** Bildet im Vergleich eine sehr kompakte, dichte Wolke im negativen Bereich beider Hauptkomponenten (unten links).
* **Fließende Übergänge:** Die Grafik bestätigt visuell genau das, was die niedrigen Silhouette-Scores aus der vorherigen Analyse mathematisch bewiesen haben: Es gibt keine harten, leeren Grenzen zwischen den Gruppen. Die Profile der Kreditnehmer gehen an den Rändern graduell und weich ineinander über.

> **Wo ist Cluster 3 (der Ausreißer)?** > Wenn, wie in k=4 Modellen üblich, ein „Cluster 3“ als extremer Ausreißer existiert (z. B. ein einzelner Antragsteller mit 117 Mio. Einkommen), fehlt dieser in dieser Ansicht. Das ist methodisch oft sinnvoll: Würde man einen derart extremen Ausreißer im Plot belassen, würde er die Skalierung der Achsen so gigantisch verzerren, dass die Cluster 0, 1 und 2 nur noch als winziger, ununterscheidbarer Punkt in der Mitte der Grafik zusammenfallen würden.

In [4]:
# Wir fixieren k bewusst auf einen fachlich interpretierbaren Wert (hier 4),
# da Silhouette und fachliche Interpretierbarkeit gegeneinander abzuwägen sind.
K = 4
labels, profiles, km, pre = cl.run_clustering(feat, k=K)
plt.show()

22:37:17 | INFO    | Clustering nutzt 12 Merkmale: ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'credit_income_ratio', 'annuity_income_ratio', 'age_years', 'employment_years', 'external_score_mean', 'BUREAU_COUNT', 'BUREAU_ACTIVE_COUNT', 'PREV_COUNT', 'INST_LATE_PAYMENT_RATIO', 'missing_values_count']
22:37:20 | INFO    | PCA-Plot: 307511 → 15000 Punkte (Subsample)
22:37:21 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\clustering\clusters_pca_2d.png
22:37:21 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\clustering\cluster_profiles.csv
22:37:21 | INFO    | Clustering fertig: k=4, erklärte Varianz (2 PCs)=35.2%
C:\Users\annis\AppData\Local\Temp\ipykernel_12020\3644668956.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Cluster-Profile & Ausfallrate je Cluster

Die Tabelle zeigt Mittelwerte je Cluster für ausgewählte Merkmale sowie die nachträglich berechnete Ausfallrate (Anteil TARGET=1). Die Profile erlauben eine inhaltliche Charakterisierung der Segmente.

**Übersicht der Ergebnisse:**

| Cluster | Größe | Ausfallrate | Einkommen | Kredit/Eink. | Ext. Score | Alter | Verspätungen |
|---|---|---|---|---|---|---|---|
| 0 | 80 761 | **6,2 %** ↓ | 153 k | 7,25 | **0,552** | 46,7 | 6,6 % |
| 1 | 69 808 | **8,9 %** ↑ | **201 k** | 3,11 | 0,483 | 45,3 | 8,1 % |
| 2 | 156 941 | **8,6 %** | 161 k | 2,64 | 0,499 | 41,9 | 7,8 % |
| 3 | 1 | **100 %** | 117 Mio | 0,005 | 0,240 | 34,5 | 37,5 % |

**Interpretation der Ausfallraten:**  
Die Ausfallraten unterscheiden sich systematisch zwischen den Clustern (6,2 % bis 8,9 % in den drei Hauptgruppen). Das belegt, dass die **rein merkmalbasierte Segmentierung ohne TARGET risikorelevante Gruppen identifiziert** – ein zentrales Ergebnis dieses Abschnitts.

**Cluster 3 – methodische Anmerkung:**  
Dieser „Cluster" besteht aus einem einzigen Antragsteller mit einem Einkommen von 117 Millionen Einheiten – ein extremer Ausreißer, der durch seine pure Distanz zu allen anderen Punkten einen eigenen Cluster bildet. Die Ausfallrate von 100 % (ein Ausfall bei einem Antragsteller) ist nicht generalisierbar. Fachlich sollte dieser Datenpunkt als Datenfehler oder Extremfall gewertet werden.

**Warum deskriptiv, nicht kausal?**  
Ein hohes `credit_income_ratio` in Cluster 0 *verursacht* keinen niedrigeren Ausfall – es kann Ausdruck einer spezifischen Kundengruppe sein (z. B. Hauskäufer mit langfristigen, gut bedienten Hypotheken). Kausalschlüsse erfordern kontrollierte Studien oder tiefergehende Domänenanalyse.

In [5]:
show_cols = ["cluster", "cluster_size", "default_rate", "AMT_INCOME_TOTAL",
             "credit_income_ratio", "external_score_mean", "age_years",
             "INST_LATE_PAYMENT_RATIO"]
show_cols = [c for c in show_cols if c in profiles.columns]
profiles[show_cols].round(3)

,cluster,cluster_size,default_rate,AMT_INCOME_TOTAL,credit_income_ratio,external_score_mean,age_years,INST_LATE_PAYMENT_RATIO
0,0,80761,0.062,1.531839e+05,7.247,0.552,46.668,0.066
1,1,69808,0.089,2.016560e+05,3.113,0.483,45.308,0.081
2,2,156941,0.086,1.614730e+05,2.641,0.499,41.863,0.078
3,3,1,1.000,1.170000e+08,0.005,0.240,34.500,0.375


## 5. Fachliche Cluster-Interpretation

Auf Basis der Cluster-Profile lassen sich drei inhaltlich klar unterscheidbare Segmente benennen (Cluster 3 ist ein Ausreißer-Cluster, kein eigenständiges Segment):

---

### Cluster 0 – „Erfahrene Hypothekenkreditnehmer" (26 %, niedrigstes Risiko)
**Profil:** Älteste Gruppe (∅ 47 Jahre), mittleres Einkommen (153 k), aber mit Abstand der höchste Kreditbetrag (1 Mio.) und das höchste Kredit-Einkommens-Verhältnis (7,25). Bester externer Score (0,552) und die wenigsten Zahlungsverspätungen (6,6 %).

**Interpretation:** Diese Gruppe nimmt sehr hohe Kredite auf – vermutlich Immobilienfinanzierungen –, bedient sie aber zuverlässig. Die Kombination aus Alter, Erfahrung und guter Bonität erklärt die **niedrigste Ausfallrate (6,2 %)** trotz des hohen absoluten Kreditbetrags.

**Risikorelevanz:** Geringrisiko-Segment. Hohe Kredit-Einkommens-Verhältnisse allein sind kein Ausfall-Prädiktor, wenn Bonität und Zahlungsdisziplin stimmen.

---

### Cluster 1 – „Einkommensstarke Mehrfach-Kreditnehmer" (23 %, höchstes Risiko)
**Profil:** Höchstes Einkommen (201 k) und längste Beschäftigung (∅ 8,7 Jahre). Gleichzeitig die meisten Bureau-Einträge (∅ 10,5) und aktiven Kredite (∅ 4,1) – deutlich mehr als alle anderen Segmente. Niedrigster externer Score (0,483) der Hauptcluster.

**Interpretation:** Paradoxerweise hat die einkommensstärkste Gruppe die **höchste Ausfallrate (8,9 %)**. Eine plausible Erklärung ist **Überschuldung**: Viele parallele Kreditverpflichtungen erhöhen das Ausfallrisiko unabhängig vom absoluten Einkommen. Der niedrige externe Score stützt diese Lesart.

**Risikorelevanz:** Hohes Risiko trotz guter Einkommenslage. Ein reines Einkommenskriterium würde dieses Segment fälschlicherweise als sicher einstufen.

---

### Cluster 2 – „Jüngere Kreditnehmer mit begrenzter Kredit-Geschichte" (51 %, moderate Risiko)
**Profil:** Größte Gruppe (157 k Antragsteller, 51 % aller Kreditnehmer). Jüngste Gruppe (∅ 42 Jahre), kürzeste Beschäftigung (∅ 5,1 Jahre), kleinste Kreditbeträge (399 k) und kaum Bureau-Vorgeschichte (∅ 2,7 Einträge, ∅ 1,0 aktive Kredite). Die meisten fehlenden Werte (31,9).

**Interpretation:** Diese Gruppe befindet sich typischerweise früh in der Kreditkarriere – erste Kredite, noch keine umfangreiche Kredithistorie. Die **Ausfallrate (8,6 %)** ist ähnlich hoch wie Cluster 1, aber aus anderem Grund: fehlende Bonität-Daten und Unerfahrenheit statt Überschuldung. Viele fehlende Werte deuten auf weniger vollständige Antragsunterlagen hin.

**Risikorelevanz:** Moderat-hohes Risiko durch Informationsmangel. Erweitertes Screening oder kleinere Anfangskredite könnten das Risiko steuern.

---

### Cluster 3 – Ausreißer (kein echtes Segment)
Ein einzelner Antragsteller mit einem Einkommen von 117 Mio. bildet durch seine extreme Distanz einen eigenen Cluster. Der Datenpunkt ist statistisch dominant und methodisch ein Artefakt – K-Means ist bekannt dafür, Ausreißer in eigene Cluster zu isolieren. Für produktive Anwendungen sollte dieser Datenpunkt als Anomalie gefiltert werden.

---

### Methodische Grenzen und Alternativen

| Schwäche K-Means | Konkrete Auswirkung hier |
|---|---|
| Kugelförmige Cluster angenommen | Cluster mit nicht-kugeliger Verteilung werden verzerrt |
| Sensibel gegenüber Ausreißern | Cluster 3 als Ausreißer-Artefakt |
| Ergebnis hängt von k-Wahl ab | k=4 fachlich begründet, nicht rein datengetrieben |
| Gleiche Clustergröße bevorzugt | Cluster 2 ist >doppelt so groß wie Cluster 1 |

**Alternativen:** Gaussian Mixture Models (GMM) erlauben flexible Clusterformen; hierarchisches Clustering benötigt kein vorher festgelegtes k; DBSCAN erkennt Ausreißer automatisch. Für eine produktive Segmentierung wäre ein Vergleich dieser Methoden empfehlenswert.